# P4 Agent 4 · Observed Duty Mapping

| 항목 | 명세 |
|---|---|
| 목적 | Validate 28 duty rows and materialize lexical top-5 candidates with unmapped preservation. |
| 담당 Agent | `P4-A4-NCS` |
| Stage ID | `A4-03-MAP-OBSERVED` |
| 입력 | `shared/handoffs/AGENT2_TO_AGENT4_DUTY_INPUT_OBSERVED_DEV.json` |
| 처리 | 28 duty schema/SHA 검증·top-5·unmapped·confidence |
| 출력 | posting_ncs_candidates and posting_ncs_matches 및 4개 종료 artifact |
| 선행 Gate | `REQUIREMENT_READY_AND_NCS_RETRIEVAL_READY` |
| 후속 활용 | NCS CSV export and Agent 2 handoff |

> Development-only orchestration. Empirical analysis and production promotion are disabled.

In [ ]:
RUN_MODE = "observed-dev"
AGENT_ID = "P4-A4-NCS"
STAGE_ID = "A4-03-MAP-OBSERVED"
CONTRACT_VERSION = "2.1.2"
SCHEMA_VERSION = "posting-ncs-candidates-v1"
DATA_VERSION = "observed-dev-20260806.1"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
AS_OF_DATE = "2026-08-06"
INPUT_MANIFEST_PATH = "shared/handoffs/AGENT2_TO_AGENT4_DUTY_INPUT_OBSERVED_DEV.json"
OUTPUT_ROOT = "ncs_mapping/data/runs/observed-dev/NCS_MAPPING_OBSERVED_20260806_01/A4-03-MAP-OBSERVED"
RANDOM_SEED = 20260806
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
PROMOTION_ALLOWED = False
DUTY_INPUT_PATH = ""
GOLD_INPUT_PATH = ""
CONTROL_SCHEMA_DIR = ""

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

NCS_ROOT = Path.cwd().resolve()
if NCS_ROOT.name != 'ncs_mapping':
    raise RuntimeError('run this notebook with cwd=ncs_mapping')
sys.path.insert(0, str(NCS_ROOT / 'src'))
assert RUN_MODE == 'observed-dev'
assert AGENT_ID == 'P4-A4-NCS' and STAGE_ID.startswith('A4-')
assert RANDOM_SEED == 20260806 and FAIL_ON_GATE is True
assert DATA_PROVENANCE == 'OBSERVED_DEVELOPMENT_ONLY'
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
resolved_duty_input = DUTY_INPUT_PATH or os.environ.get('P4_A2_DUTY_HANDOFF', '')
resolved_gold_input = GOLD_INPUT_PATH or os.environ.get('P4_NCS_GOLD_INPUT', '')
resolved_schema_dir = CONTROL_SCHEMA_DIR or os.environ.get('P4_CONTROL_SCHEMA_DIR', '')

In [ ]:
from p4_ncs.contracts.observed_duty import load_and_validate_observed_duties
from p4_ncs.dictionary.alias_dictionary import load_alias_dictionary
from p4_ncs.mapping.observed_baseline import map_observed_duties
from p4_ncs.retrieval.lexical_index import LexicalIndex

if not resolved_duty_input:
    raise FileNotFoundError('DUTY_INPUT_PATH or P4_A2_DUTY_HANDOFF is required')
duties, duty_validation, duty_envelope = load_and_validate_observed_duties(resolved_duty_input)
ncs_units = pd.read_parquet(NCS_ROOT / 'data/processed/ncsUnit.parquet')
codeset = pd.read_parquet(NCS_ROOT / 'data/processed/coreAiItCodeSet.parquet')
aliases = load_alias_dictionary(NCS_ROOT / 'configs/ncs_alias_dictionary.yaml')
candidates_preview, matches_preview = map_observed_duties(duties, LexicalIndex.build(ncs_units, codeset), aliases, codeset, DATA_VERSION, top_k=5)
input_audit = {'dutyRows': duty_validation.row_count, 'candidateRows': len(candidates_preview), 'matchRows': len(matches_preview), 'maxTopK': int(candidates_preview.groupby('sectionId').size().max()), 'unmappedRows': int(matches_preview['ncsSubCode'].isna().sum()), 'denseAllNull': bool(matches_preview['denseScore'].isna().all()), 'goldValidatedAny': bool(matches_preview['goldValidatedFlag'].any())}
assert input_audit['dutyRows'] == 28 and input_audit['maxTopK'] <= 5
assert input_audit['denseAllNull'] and not input_audit['goldValidatedAny']
input_audit

In [ ]:
from p4_ncs.workflow.observed import run_stage

stage_manifest = run_stage('A4-03-MAP-OBSERVED', root=NCS_ROOT, duty_input_path=resolved_duty_input or None, gold_input_path=resolved_gold_input or None, schema_dir=resolved_schema_dir or None)
stage_manifest

In [ ]:
stage_root = NCS_ROOT / 'data/runs' / RUN_MODE / 'NCS_MAPPING_OBSERVED_20260806_01' / stage_manifest['stageId']
expected_artifacts = {'stage_manifest.json', 'stage_metrics.json', 'stage_quality.csv', 'CHECKSUMS.sha256'}
actual_artifacts = {path.name for path in stage_root.iterdir() if path.is_file()}
assert actual_artifacts == expected_artifacts
termination_summary = {'stageId': stage_manifest['stageId'], 'status': stage_manifest['status'], 'rowCounts': stage_manifest['rowCounts'], 'artifacts': sorted(actual_artifacts)}
termination_summary